In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
df = {}

In [3]:
df["train_orig"] = pd.read_csv("train.csv")
df["test_orig"]  = pd.read_csv("test.csv")

In [4]:
df["total_orig"] = pd.concat(objs=[df["train_orig"], df["test_orig"]], ignore_index=True)

In [5]:
df["total_orig"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     891 non-null    float64
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   object 
 4   Sex          1309 non-null   object 
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   object 
 9   Fare         1308 non-null   float64
 10  Cabin        295 non-null    object 
 11  Embarked     1307 non-null   object 
dtypes: float64(3), int64(4), object(5)
memory usage: 122.8+ KB


In [6]:
incomplete_columns = ["Cabin"]

df["total"] = df["total_orig"].drop(labels=incomplete_columns, axis=1)

In [7]:
unhelpful_columns = ["PassengerId", "Name", "Ticket"]

df["total"] = df["total"].drop(labels=unhelpful_columns, axis=1)

In [8]:
# drop_na_columns = ["Embarked"]
drop_na_columns = []

for column in drop_na_columns:
    df["total"] = df["total"][~df["total"][column].isna()]

In [9]:
df["total"] = pd.get_dummies(df["total"], columns=["Sex", "Embarked"])

In [10]:
def run_clf(clf, column = "Survived", drop_na_columns=False):
    df_total = globals()["df"]["total"]
    columns_with_na = df_total.columns[df_total.isna().any()]
    train    = df_total[~df_total[column].isna()]
    test     = df_total[ df_total[column].isna()].drop(column, axis=1)

    if drop_na_columns:        
        columns_with_na = test.columns[test.isna().any()]
        train = train.drop(columns_with_na, axis=1)
        test  = test.drop(columns_with_na, axis=1)
    else:
        train = train.dropna()
        test  = test.dropna()

    X_train = train.drop(column, axis=1)
    y_train = train[column]
    X_test  = test.copy()
    
    if X_test.shape[0] == 0: return []

    computed_column_name = column + "_computed"
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    df_total[computed_column_name] = df_total[column]
#     import IPython; IPython.embed(using=None)
    df_total.loc[X_test.index, computed_column_name] = y_pred
    df_total.drop(column, axis=1, inplace=True)

    acc_log = round(clf.score(X_train, y_train) * 100, 2)
    print(acc_log)
    return y_pred

In [11]:
df_total = df["total"]

In [12]:
df_total.columns[df_total.isna().any()]

Index(['Survived', 'Age', 'Fare'], dtype='object')

In [13]:
df["total"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Survived    891 non-null    float64
 1   Pclass      1309 non-null   int64  
 2   Age         1046 non-null   float64
 3   SibSp       1309 non-null   int64  
 4   Parch       1309 non-null   int64  
 5   Fare        1308 non-null   float64
 6   Sex_female  1309 non-null   uint8  
 7   Sex_male    1309 non-null   uint8  
 8   Embarked_C  1309 non-null   uint8  
 9   Embarked_Q  1309 non-null   uint8  
 10  Embarked_S  1309 non-null   uint8  
dtypes: float64(3), int64(3), uint8(5)
memory usage: 67.9 KB


In [14]:
import xgboost as xgb

run_clf(xgb.XGBRegressor(), "Age")
run_clf(xgb.XGBRegressor(), "Age_computed", drop_na_columns=True)

76.82
72.8


array([27.727854, 37.551636, 16.021809, 17.849596, 24.921629, 29.20455 ,
       47.887882, 43.317356, 31.333317, 26.262447, 30.27901 , 30.104246,
       27.727854, 46.18832 , 23.834454, 23.234638, 27.512342, 30.104246,
       43.317356, 43.317356, 29.687988, 19.632296, 34.338455, 38.985527,
       43.317356,  9.39638 , 39.50154 , 26.269484, 47.77787 , 47.887882,
       31.107533, 16.909748, 36.164795, 42.67603 , 33.508835, 25.309826,
       43.317356, 24.660961, 48.50802 , 20.533846, 26.09694 , 35.66128 ,
       27.44459 , 19.756197, 30.104246, 27.053837, 23.234638, 30.30994 ,
       25.852894, 20.588537, 17.828932, 33.508835, 43.317356, 27.727854,
       40.87597 , 33.508835, 24.921629, 43.317356, 28.31533 , 29.542662,
       23.234638, 30.698467, 25.309826, 30.104246, 43.026073, 25.309826,
       16.021809, 30.749619, 23.234638, 32.404823, 29.542662, 25.309826,
       45.388836, 17.078188, 25.852894, 43.317356, 45.274094, 25.665613,
       43.317356, 34.00158 , 34.12172 , 18.299953, 

In [15]:
df["total"]

,Survived,Pclass,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Age_computed_computed
0,0.0,3,1,0,7.2500,0,1,0,0,1,22.000000
1,1.0,1,1,0,71.2833,1,0,1,0,0,38.000000
2,1.0,3,0,0,7.9250,1,0,0,0,1,26.000000
3,1.0,1,1,0,53.1000,1,0,0,0,1,35.000000
4,0.0,3,0,0,8.0500,0,1,0,0,1,35.000000
...,...,...,...,...,...,...,...,...,...,...,...
1304,NaN,3,0,0,8.0500,0,1,0,0,1,30.104246
1305,NaN,1,0,0,108.9000,1,0,1,0,0,39.000000
1306,NaN,3,0,0,7.2500,0,1,0,0,1,38.500000
1307,NaN,3,0,0,8.0500,0,1,0,0,1,30.104246


In [16]:
df["total"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Survived               891 non-null    float64
 1   Pclass                 1309 non-null   int64  
 2   SibSp                  1309 non-null   int64  
 3   Parch                  1309 non-null   int64  
 4   Fare                   1308 non-null   float64
 5   Sex_female             1309 non-null   uint8  
 6   Sex_male               1309 non-null   uint8  
 7   Embarked_C             1309 non-null   uint8  
 8   Embarked_Q             1309 non-null   uint8  
 9   Embarked_S             1309 non-null   uint8  
 10  Age_computed_computed  1309 non-null   float64
dtypes: float64(3), int64(3), uint8(5)
memory usage: 67.9 KB


In [17]:
run_clf(xgb.XGBRegressor(), "Fare")
run_clf(xgb.XGBRegressor(), "Fare", drop_na_columns=True)

91.94


array([7.0019403], dtype=float32)

In [18]:
df["total"][df["total_orig"].Embarked.isna()]

,Survived,Pclass,SibSp,Parch,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Age_computed_computed,Fare_computed
61,1.0,1,0,0,1,0,0,0,0,38.0,80.0
829,1.0,1,0,0,1,0,0,0,0,62.0,80.0


In [19]:
df["total"].info

<bound method DataFrame.info of       Survived  Pclass  SibSp  Parch  Sex_female  Sex_male  Embarked_C  \
0          0.0       3      1      0           0         1           0   
1          1.0       1      1      0           1         0           1   
2          1.0       3      0      0           1         0           0   
3          1.0       1      1      0           1         0           0   
4          0.0       3      0      0           0         1           0   
...        ...     ...    ...    ...         ...       ...         ...   
1304       NaN       3      0      0           0         1           0   
1305       NaN       1      0      0           1         0           1   
1306       NaN       3      0      0           0         1           0   
1307       NaN       3      0      0           0         1           0   
1308       NaN       3      1      1           0         1           1   

      Embarked_Q  Embarked_S  Age_computed_computed  Fare_computed  
0         

In [20]:
df["total"]

,Survived,Pclass,SibSp,Parch,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Age_computed_computed,Fare_computed
0,0.0,3,1,0,0,1,0,0,1,22.000000,7.2500
1,1.0,1,1,0,1,0,1,0,0,38.000000,71.2833
2,1.0,3,0,0,1,0,0,0,1,26.000000,7.9250
3,1.0,1,1,0,1,0,0,0,1,35.000000,53.1000
4,0.0,3,0,0,0,1,0,0,1,35.000000,8.0500
...,...,...,...,...,...,...,...,...,...,...,...
1304,NaN,3,0,0,0,1,0,0,1,30.104246,8.0500
1305,NaN,1,0,0,1,0,1,0,0,39.000000,108.9000
1306,NaN,3,0,0,0,1,0,0,1,38.500000,7.2500
1307,NaN,3,0,0,0,1,0,0,1,30.104246,8.0500


In [21]:
from sklearn.ensemble import RandomForestClassifier

run_clf(RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1), "Survived")

86.31


array([0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 1., 0., 1., 1., 0.,
       0., 0., 0., 0., 1., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1., 1., 0.,
       0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1., 1., 1., 0.,
       0., 1., 1., 0., 0., 0., 1., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0.,
       0., 1., 1., 1., 1., 1., 0., 1., 0., 0., 0., 1., 0., 1., 0., 1., 0.,
       0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0.,
       1., 1., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       1., 0., 0., 1., 1., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 1., 1., 0., 1., 1., 0., 0., 1., 0., 1., 0., 1.,
       0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 1., 0., 0., 1., 1., 0., 1.,
       0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 1.,
       0., 1., 0., 1., 1.

In [22]:
df["total"]["Survived_computed"] = df["total"]["Survived_computed"].round().astype(int)

In [23]:
df["total"]

,Pclass,SibSp,Parch,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Age_computed_computed,Fare_computed,Survived_computed
0,3,1,0,0,1,0,0,1,22.000000,7.2500,0
1,1,1,0,1,0,1,0,0,38.000000,71.2833,1
2,3,0,0,1,0,0,0,1,26.000000,7.9250,1
3,1,1,0,1,0,0,0,1,35.000000,53.1000,1
4,3,0,0,0,1,0,0,1,35.000000,8.0500,0
...,...,...,...,...,...,...,...,...,...,...,...
1304,3,0,0,0,1,0,0,1,30.104246,8.0500,0
1305,1,0,0,1,0,1,0,0,39.000000,108.9000,1
1306,3,0,0,0,1,0,0,1,38.500000,7.2500,0
1307,3,0,0,0,1,0,0,1,30.104246,8.0500,0


In [24]:
df["total_orig"]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
1304,1305,NaN,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
1305,1306,NaN,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
1306,1307,NaN,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
1307,1308,NaN,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [25]:
df["total"]["PassengerId"] = df["total_orig"]["PassengerId"]

In [26]:
test_orig = df["total_orig"][df["total_orig"]["Survived"].isna()]
df["total"].loc[test_orig.index][["PassengerId", "Survived_computed"]].to_csv(
    "submission.csv",
    header=["PassengerId", "Survived"],
    index=False
)